In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%writefile models.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

class AttentionDoubleLSTM(nn.Module):
    def __init__(self, input_dim=14, hidden_dim=128, num_layers=2):
        super().__init__()
        self.projection = nn.Linear(input_dim, hidden_dim)
        self.ln_proj = nn.LayerNorm(hidden_dim)
        self.lstm = nn.LSTM(input_size=hidden_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=0.1)
        self.ln_lstm = nn.LayerNorm(hidden_dim)
        self.attention_fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # x shape: [batch_size, seq_len, input_dim]
        e_t = F.relu(self.ln_proj(self.projection(x)))
        lstm_out, _ = self.lstm(e_t)
        lstm_out = self.ln_lstm(lstm_out)
        attn_scores = self.attention_fc(lstm_out) # [batch_size, seq_len, 1]
        attn_weights = F.softmax(attn_scores, dim=1) # [batch_size, seq_len, 1]
        c_ctx = torch.sum(attn_weights * lstm_out, dim=1) # [batch_size, hidden_dim]
        return c_ctx

class PPOAgent(nn.Module):
    def __init__(self, input_dim=14, hidden_dim=128):
        super().__init__()
        self.feature_extractor = AttentionDoubleLSTM(input_dim, hidden_dim)
        
        # Nhánh backbone độc lập cho Critic và Actor để chuyên môn hóa đặc trưng
        self.critic_backbone = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim)
        )
        self.actor_backbone = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim)
        )
        
        self.critic = nn.Linear(hidden_dim, 1)
        self.actor_targ = nn.Linear(hidden_dim, 4)
        self.actor_lr   = nn.Linear(hidden_dim, 3)
        self.actor_mult = nn.Linear(hidden_dim, 3)
        self.actor_enh  = nn.Linear(hidden_dim, 3)

    def forward(self, state):
        c_ctx = self.feature_extractor(state)
        
        # Nhánh tính toán Value
        val_features = self.critic_backbone(c_ctx)
        value = self.critic(val_features)
        
        # Nhánh tính toán Policy
        act_features = self.actor_backbone(c_ctx)
        logits_list = [
            self.actor_targ(act_features),
            self.actor_lr(act_features),
            self.actor_mult(act_features),
            self.actor_enh(act_features)
        ]
        return value, logits_list

    def get_action(self, state, action=None):
        value, logits_list = self.forward(state)
        actions, log_probs, entropies = [], [], []
        for i, logits in enumerate(logits_list):
            dist = Categorical(logits=logits)
            act = dist.sample() if action is None else action[:, i]
            actions.append(act)
            log_probs.append(dist.log_prob(act))
            entropies.append(dist.entropy())
        return torch.stack(actions, dim=-1), torch.stack(log_probs, dim=-1).sum(dim=-1), entropies, value


In [ ]:
%%writefile inference.py
import torch
import numpy as np
from models import PPOAgent
from collections import deque

class RealTimeAutoscaler:
    def __init__(self, model_path="ppo_double_lstm_scaler.pth"):
        # 1. Khởi tạo lại cấu trúc bộ não mạng Neural đã cải tiến
        self.agent = PPOAgent(input_dim=14, hidden_dim=128)
        
        # 2. Nạp trọng số đã được tối ưu hóa vào
        self.agent.load_state_dict(torch.load(model_path, map_location=torch.device('cpu'), weights_only=True))
        self.agent.eval()  # Chuyển sang chế độ suy luận (Inference Mode)
        
        # 3. Cửa sổ trượt w=3 ghi nhớ lịch sử metrics
        self.state_buffer = deque(maxlen=3)
        for _ in range(3):
            self.state_buffer.append(np.zeros(14))

    def predict_next_action(self, live_metrics_14d):
        """
        Hàm nhận vào vector metrics 14 chiều lấy trực tiếp theo thời gian thực
        và trả về các quyết định cấu hình điều phối tài nguyên chi tiết.
        """
        # Đẩy trượt thông số mới vào bộ đệm
        self.state_buffer.append(live_metrics_14d)
        
        # Khôi phục cấu trúc ma trận Tensor chuẩn (1, 3, 14)
        state_tensor = torch.FloatTensor(np.array(self.state_buffer)).unsqueeze(0)
        
        # Thực hiện suy luận không tính toán đạo hàm
        with torch.no_grad():
            action, _, _, _ = self.agent.get_action(state_tensor)
            
        # Giải mã 4 hành động cấu hình hạ tầng
        a_targ, a_lr, a_mult, a_enh = action[0].tolist()
        
        # Map sang giá trị cấu hình vật lý thực tế
        target_cpu = {0: 30, 1: 50, 2: 70, 3: 90}[a_targ]
        cooldown_steps = {0: 1, 1: 3, 2: 5}[a_lr]
        multiplier = {0: 1.0, 1: 1.5, 2: 2.0}[a_mult]
        enh_mode = {0: "Normal", 1: "Aggressive Scale-up", 2: "Cost-Saving Scale-down"}[a_enh]
        
        return {
            "target_cpu": target_cpu,
            "cooldown_steps": cooldown_steps,
            "multiplier": multiplier,
            "enh_mode": enh_mode
        }

# =========================================================
# VÒNG LẶP ĐIỀU KHIỂN HẠ TẦNG THỰC TẾ (CONTROL LOOP AGENT)
# =========================================================
if __name__ == "__main__":
    import os
    
    model_file = "ppo_double_lstm_scaler.pth"
    if not os.path.exists(model_file):
        print(f"[Error] Model weight file '{model_file}' not found. Please run 'python train.py' first to train the model!")
        exit(1)
        
    # Khởi tạo bộ điều phối cấu hình
    scaler = RealTimeAutoscaler(model_path=model_file)
    
    print("=== AI AUTOSCALER READY FOR PRODUCTION ===")
    
    # Giả lập vòng lặp quét thông số định kỳ hệ thống
    for step in range(1, 6):
        # 1. Đo đạc thông số thực tế từ máy chủ (đầu vào 14 chiều giả lập theo định dạng giàu trạng thái)
        # s_t[0]: workload_demand, s_t[1]: cpu_util, s_t[2]: replica_ratio, s_t[3]: target_ratio, ...
        live_sampled_metrics = np.zeros(14)
        live_sampled_metrics[0] = np.random.uniform(0.2, 0.8) # Workload demand
        live_sampled_metrics[1] = np.random.uniform(0.3, 0.9) # CPU util
        live_sampled_metrics[2] = 0.3                         # Replica ratio (3/10)
        live_sampled_metrics[3] = 0.5                         # Target ratio (50%)
        live_sampled_metrics[7] = 0.6                         # RAM
        live_sampled_metrics[8] = 0.15                        # Response Time
        
        # 2. Đưa vào cho mô hình AI dự đoán hành động cấu hình tối ưu
        decisions = scaler.predict_next_action(live_sampled_metrics)
        
        # 3. In ra quyết định điều phối hạ tầng
        print(f"\n[Step {step}] Received metrics from Prometheus...")
        print(f" -> Workload: {live_sampled_metrics[0]*100:.1f}% | Actual CPU: {live_sampled_metrics[1]*100:.1f}%")
        print(f" -> AI Decision:")
        print(f"    - Target CPU utilization: {decisions['target_cpu']}%")
        print(f"    - Cooldown: {decisions['cooldown_steps']} steps")
        print(f"    - Scale multiplier: {decisions['multiplier']}x")
        print(f"    - Operating mode: {decisions['enh_mode']}")
        
        # Trong thực tế, bạn sẽ chèn lệnh gọi API hạ tầng ở đây:
        # e.g. client.patch_namespaced_horizontal_pod_autoscaler(...)


In [ ]:
%%writefile environment.py
import numpy as np
import torch
import pandas as pd
from collections import deque
import math
import os

class KubernetesEnv:
    def __init__(self, csv_path, max_steps=500):
        self.max_steps = max_steps
        self.csv_path = csv_path
        
        # Cơ chế fallback tự động tạo dữ liệu nếu không tìm thấy file dataset của Colab
        if not os.path.exists(csv_path):
            print(f"[Warning] Dataset not found at '{csv_path}'. Generating synthetic CPU workload for local run...")
            # Tạo dữ liệu workload mô phỏng 20,000 bước thời gian (gồm dao động ngày đêm và nhiễu)
            np.random.seed(42)
            steps = 20000
            time = np.linspace(0, 40 * np.pi, steps)
            # Tải nền + Dao động tuần hoàn + Dao động đột biến nhỏ + Nhiễu Gaussian
            workload = 0.5 + 0.3 * np.sin(time) + 0.1 * np.sin(time / 4) + np.random.normal(0, 0.04, steps)
            workload = np.clip(workload, 0.05, 0.95)
            self.df = pd.DataFrame({'value': workload})
        else:
            self.df = pd.read_csv(csv_path)
            
        self.total_rows = len(self.df)
        self.current_row = 0
        self.state_buffer = deque(maxlen=3)
        
        # Các biến trạng thái của hạ tầng ảo hóa
        self.cpu_target = 50.0       # Ngưỡng CPU Target cấu hình (%)
        self.replicas = 3            # Số lượng Pod/VM hoạt động (1 -> 10)
        self.min_replicas = 1
        self.max_replicas = 10
        self.cooldown_remaining = 0  # Số bước còn lại trong thời gian chờ ổn định (cooldown)
        self.prev_replicas = 3       # Lưu số lượng replicas bước trước để tính dao động
        self.last_action_idx = [1, 0, 0, 0] # Lưu chỉ số hành động gần nhất

    def reset(self):
        self.current_step = 0
        self.replicas = 3
        self.cpu_target = 50.0
        self.cooldown_remaining = 0
        self.prev_replicas = 3
        self.last_action_idx = [1, 0, 0, 0]
        
        if self.current_row + self.max_steps >= self.total_rows: 
            self.current_row = 0
            
        # Làm sạch và điền trước bộ đệm 3 bước bằng vector 0
        self.state_buffer.clear()
        for _ in range(3): 
            self.state_buffer.append(np.zeros(14))
            
        return self._get_tensor_state()

    def _build_state_vector(self, workload_val):
        """
        Xây dựng vector trạng thái 14 chiều phong phú từ các thông số thực tế của cụm máy chủ:
        s_t[0]: Nhu cầu workload thô từ CSV (0.0 -> 1.0)
        s_t[1]: CPU utilization thực tế của hệ thống (workload / replicas) (0.0 -> 1.0)
        s_t[2]: Tỷ lệ số replica hiện tại (replicas / max_replicas)
        s_t[3]: Ngưỡng target CPU cài đặt (cpu_target / 100.0)
        s_t[4]: Hướng điều chỉnh tài nguyên ở bước trước (-1.0: giảm, 0.0: giữ nguyên, 1.0: tăng)
        s_t[5]: Tỷ lệ thời gian cooldown còn lại (cooldown_remaining / max_cooldown)
        s_t[6]: Tốc độ thay đổi CPU utilization so với bước trước
        s_t[7]: Bộ nhớ Ram giả lập (tỷ lệ thuận với CPU kèm nhiễu đo đạc)
        s_t[8]: Chỉ số độ trễ phản hồi (Response Time) (tăng phi tuyến theo lý thuyết hàng đợi M/M/1)
        s_t[9]: Trạng thái vi phạm SLA (1.0 nếu CPU thực tế > 90%, ngược lại 0.0)
        s_t[10]: Thành phần Sin của thời gian (biểu thị tính tuần hoàn trong ngày)
        s_t[11]: Thành phần Cos của thời gian
        s_t[12]: Lưu lượng QPS (truy vấn/giây) giả lập tỷ lệ thuận với workload
        s_t[13]: Lịch sử chỉ số hệ số nhân (multiplier) hành động trước
        """
        s_t = np.zeros(14)
        s_t[0] = workload_val
        
        # CPU thực tế tải trên các pod. Giả định 4.0 đơn vị workload là đầy tải cho 4 replicas
        actual_cpu = min(1.0, (workload_val * 4.0) / max(1.0, self.replicas))
        s_t[1] = actual_cpu
        s_t[2] = self.replicas / self.max_replicas
        s_t[3] = self.cpu_target / 100.0
        
        # Hướng scaling gần nhất
        if self.replicas > self.prev_replicas:
            s_t[4] = 1.0
        elif self.replicas < self.prev_replicas:
            s_t[4] = -1.0
        else:
            s_t[4] = 0.0
            
        s_t[5] = self.cooldown_remaining / 5.0 # Chuẩn hóa với cooldown tối đa = 5 bước
        
        # Tốc độ thay đổi CPU
        if len(self.state_buffer) > 0:
            s_t[6] = actual_cpu - self.state_buffer[-1][1]
        else:
            s_t[6] = 0.0
            
        # Bộ nhớ Ram giả lập (tải CPU cao thường kéo theo RAM cao)
        s_t[7] = min(0.95, workload_val * 0.75 + 0.15 + np.random.normal(0, 0.02))
        
        # Độ trễ phản hồi hệ thống (M/M/1 queue model: Latency = 1 / (1 - CPU))
        if actual_cpu >= 0.95:
            latency = 20.0
        else:
            latency = 1.0 / (1.0 - actual_cpu + 1e-5)
        s_t[8] = min(1.0, latency / 20.0) # chuẩn hóa về [0, 1]
        
        # Trạng thái SLA
        s_t[9] = 1.0 if actual_cpu > 0.90 else 0.0
        
        # Tín hiệu thời gian tuần hoàn
        s_t[10] = math.sin(self.current_step * 2 * math.pi / 100.0)
        s_t[11] = math.cos(self.current_step * 2 * math.pi / 100.0)
        s_t[12] = workload_val
        s_t[13] = self.last_action_idx[2] / 2.0 # Chuẩn hóa a_mult (0, 1, 2)
        
        return s_t

    def _get_tensor_state(self):
        return torch.FloatTensor(np.array(self.state_buffer)).unsqueeze(0)

    def step(self, actions):
        self.current_step += 1
        
        # Giải mã 4 hành động rời rạc từ Multi-Discrete Actor
        # actions shape: [batch, 4]
        a_targ = actions[0][0].item()
        a_lr   = actions[0][1].item()
        a_mult = actions[0][2].item()
        a_enh  = actions[0][3].item()
        
        self.last_action_idx = [a_targ, a_lr, a_mult, a_enh]
        
        # 1. Cấu hình Target CPU
        self.cpu_target = {0: 30.0, 1: 50.0, 2: 70.0, 3: 90.0}[a_targ]
        
        # 2. Đọc workload từ tập dữ liệu
        row_data = self.df.iloc[self.current_row]
        workload_val = float(row_data['value'])
        self.current_row += 1
        if self.current_row >= self.total_rows: 
            self.current_row = 0
            
        # 3. Tính toán CPU trước khi scaling
        current_cpu = min(1.0, (workload_val * 4.0) / max(1.0, self.replicas))
        
        # 4. Thuật toán HPA ước lượng replica mong muốn: Desired = ceil(Current * (Current_CPU / Target_CPU))
        target_ratio = self.cpu_target / 100.0
        desired_replicas = math.ceil(self.replicas * (current_cpu / target_ratio))
        desired_replicas = max(self.min_replicas, min(self.max_replicas, desired_replicas))
        
        self.prev_replicas = self.replicas
        
        # Quản lý cooldown
        if self.cooldown_remaining > 0:
            self.cooldown_remaining -= 1
            
        # 5. Thực thi logic scaling có ảnh hưởng bởi a_mult, a_lr, a_enh
        if desired_replicas > self.replicas:
            # TĂNG TÀI NGUYÊN (SCALE UP)
            mult_factor = {0: 1.0, 1: 1.5, 2: 2.0}[a_mult]
            diff = desired_replicas - self.replicas
            scale_up_amount = math.ceil(diff * mult_factor)
            
            # Chế độ tăng cường a_enh = 1 (Tấn công - Aggressive) hoặc khi CPU quá cao
            if a_enh == 1 or current_cpu > 0.85:
                scale_up_amount = max(scale_up_amount, 2)
                
            self.replicas = min(self.max_replicas, self.replicas + scale_up_amount)
            
            # Kích hoạt cooldown
            cooldown_steps = {0: 1, 1: 3, 2: 5}[a_lr]
            self.cooldown_remaining = cooldown_steps
            
        elif desired_replicas < self.replicas:
            # GIẢM TÀI NGUYÊN (SCALE DOWN)
            if self.cooldown_remaining == 0:
                # Chế độ tối ưu chi phí a_enh = 2 (Cost-saving) cho phép thu hồi nhanh
                if a_enh == 2:
                    scale_down_amount = self.replicas - desired_replicas
                else:
                    scale_down_amount = 1 # chế độ mặc định chỉ thu hồi từ từ từng replica để tránh dao động
                    
                self.replicas = max(self.min_replicas, self.replicas - scale_down_amount)
                
                # Kích hoạt cooldown
                cooldown_steps = {0: 1, 1: 3, 2: 5}[a_lr]
                self.cooldown_remaining = cooldown_steps
                
        # 6. Cập nhật trạng thái mới sau khi scale
        new_s_t = self._build_state_vector(workload_val)
        self.state_buffer.append(new_s_t)
        
        # 7. Tính toán Reward đa mục tiêu
        new_cpu = new_s_t[1]
        
        # Phạt SLA khi CPU quá tải (>90% làm tăng đáng kể tỉ lệ drop request/response time)
        sla_penalty = 0.0
        if new_cpu > 0.90:
            sla_penalty = 12.0 * (new_cpu - 0.90)
            
        # Phạt chi phí sử dụng tài nguyên (khuyến khích giữ ít replica nhất có thể)
        cost_penalty = 0.15 * self.replicas
        
        # Phạt dao động hạ tầng (phạt mỗi khi thay đổi số lượng replica)
        action_penalty = 0.0
        if self.replicas != self.prev_replicas:
            action_penalty = 0.3
            
        # Thưởng khi duy trì CPU trong vùng tối ưu hiệu năng và năng lượng (45% -> 75%)
        if 0.45 <= new_cpu <= 0.75:
            util_reward = 1.0
        else:
            dist = min(abs(new_cpu - 0.45), abs(new_cpu - 0.75))
            util_reward = math.exp(-dist / 0.25)
            
        reward = util_reward - sla_penalty - cost_penalty - action_penalty
        
        return self._get_tensor_state(), reward, self.current_step >= self.max_steps


In [ ]:
%%writefile train.py
import torch
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.optim.lr_scheduler import CosineAnnealingLR
from models import PPOAgent
from environment import KubernetesEnv

# HYPERPARAMETERS
LEARNING_RATE = 2e-4
GAMMA = 0.99
GAE_LAMBDA = 0.93
CLIP_EPSILON = 0.2
ENTROPY_COEFF = 0.01
UPDATE_EPOCHS = 10
BATCH_SIZE = 128
ROLLOUT_STEPS = 512
NUM_EPISODES = 1000  # Cấu hình 1000 vòng lặp cho giai đoạn train thật

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training device: {device}")

env = KubernetesEnv(csv_path="/content/drive/MyDrive/Dataset/data.csv", max_steps=ROLLOUT_STEPS)
agent = PPOAgent(input_dim=14, hidden_dim=128).to(device)
optimizer = optim.Adam(agent.parameters(), lr=LEARNING_RATE)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPISODES, eta_min=1e-5)

def train():
    # Khởi tạo các mảng lưu lịch sử để phục vụ vẽ đồ thị báo cáo
    reward_history = []
    loss_history = []
    actor_loss_history = []
    critic_loss_history = []
    entropy_history = []
    
    print("=== STARTING TRAINING PROCESS WITH IMPROVED CONFIGURATION ===")
    for episode in range(1, NUM_EPISODES + 1):
        state = env.reset().to(device)
        states, actions, log_probs, rewards, dones, values = [], [], [], [], [], []
        
        # PHA 1: THU THẬP TRẢI NGHIỆM (ROLLOUT)
        for _ in range(ROLLOUT_STEPS):
            with torch.no_grad():
                action, log_prob, _, value = agent.get_action(state)
                
            next_state, reward, done = env.step(action)
            next_state = next_state.to(device)
            
            states.append(state)
            actions.append(action)
            log_probs.append(log_prob)
            rewards.append(reward)
            dones.append(done)
            values.append(value)
            
            state = next_state
            if done:
                break

        states = torch.cat(states, dim=0)
        actions = torch.cat(actions, dim=0)
        log_probs = torch.cat(log_probs, dim=0)
        values = torch.cat(values, dim=0).squeeze()
        
        # PHA 2: TÍNH TOÁN LỢI THẾ GAE
        returns = np.zeros_like(rewards)
        advantages = np.zeros_like(rewards)
        gae = 0
        next_value = 0 if dones[-1] else values[-1].item()
        
        for t in reversed(range(len(rewards))):
            delta = rewards[t] + GAMMA * next_value * (1 - dones[t]) - values[t].item()
            gae = delta + GAMMA * GAE_LAMBDA * (1 - dones[t]) * gae
            advantages[t] = gae
            returns[t] = advantages[t] + values[t].item()
            next_value = values[t].item()
            
        advantages = torch.FloatTensor(advantages).to(device)
        returns = torch.FloatTensor(returns).to(device)
        # Chuẩn hóa Advantages
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        # PHA 3: TỐI ƯU HÓA TRỌNG SỐ PPO BẰNG MINI-BATCH
        epoch_losses = []
        epoch_actor_losses = []
        epoch_critic_losses = []
        epoch_entropies = []
        
        dataset_size = len(rewards)
        indices = np.arange(dataset_size)
        
        for _ in range(UPDATE_EPOCHS):
            # Trộn dữ liệu ngẫu nhiên trước mỗi Epoch cập nhật
            np.random.shuffle(indices)
            
            # Chia nhỏ Rollout thành các mini-batch
            for start_idx in range(0, dataset_size, BATCH_SIZE):
                batch_indices = indices[start_idx:start_idx + BATCH_SIZE]
                if len(batch_indices) < 32:  # Bỏ qua các batch quá nhỏ ở cuối để tránh nhiễu nhiễu đạo hàm
                    continue
                    
                mb_states = states[batch_indices]
                mb_actions = actions[batch_indices]
                mb_log_probs = log_probs[batch_indices]
                mb_advantages = advantages[batch_indices]
                mb_returns = returns[batch_indices]
                
                # Tính giá trị dự báo mới của chính sách hiện tại
                _, new_log_probs, entropies, new_values = agent.get_action(mb_states, mb_actions)
                new_values = new_values.squeeze()
                
                # Tỷ lệ chính sách mới / cũ
                ratio = torch.exp(new_log_probs - mb_log_probs)
                
                # Cắt (clip) hàm mục tiêu Actor
                surr1 = ratio * mb_advantages
                surr2 = torch.clamp(ratio, 1.0 - CLIP_EPSILON, 1.0 + CLIP_EPSILON) * mb_advantages
                actor_loss = -torch.min(surr1, surr2).mean()
                
                # Hàm Critic Loss (MSE)
                critic_loss = F.mse_loss(new_values, mb_returns)
                
                # Entropy khuyến khích khám phá hành động mới
                total_entropy = torch.stack(entropies).sum(dim=0).mean()
                
                # Hàm loss tổng hợp của PPO
                loss = actor_loss + 0.5 * critic_loss - ENTROPY_COEFF * total_entropy
                
                optimizer.zero_grad()
                loss.backward()
                # Cắt bớt Gradient (Clipping) bảo vệ mạng LSTM khỏi bùng nổ đạo hàm
                torch.nn.utils.clip_grad_norm_(agent.parameters(), max_norm=0.5)
                optimizer.step()
                
                epoch_losses.append(loss.item())
                epoch_actor_losses.append(actor_loss.item())
                epoch_critic_losses.append(critic_loss.item())
                epoch_entropies.append(total_entropy.item())
                
        # Cập nhật tốc độ học theo scheduler
        scheduler.step()
        
        # Lưu lại lịch sử
        reward_history.append(sum(rewards))
        loss_history.append(np.mean(epoch_losses))
        actor_loss_history.append(np.mean(epoch_actor_losses))
        critic_loss_history.append(np.mean(epoch_critic_losses))
        entropy_history.append(np.mean(epoch_entropies))
        
        if episode % 10 == 0 or episode == 1:
            curr_lr = optimizer.param_groups[0]['lr']
            print(f"Episode {episode:04d}/{NUM_EPISODES} | "
                  f"Reward: {sum(rewards):.2f} | "
                  f"Loss: {np.mean(epoch_losses):.4f} | "
                  f"Actor Loss: {np.mean(epoch_actor_losses):.4f} | "
                  f"Critic Loss: {np.mean(epoch_critic_losses):.4f} | "
                  f"LR: {curr_lr:.6f}")

    # ==========================================
    # PHẦN XUẤT FILE ĐỒ THỊ VÀ MÔ HÌNH SAU KHI HOÀN THÀNH
    # ==========================================
    print("\n=== TRAINING COMPLETED! EXPORTING ARTIFACTS ===")
    
    # 1. Lưu cấu trúc trọng số mạng neural đã cải tiến
    torch.save(agent.state_dict(), "ppo_double_lstm_scaler.pth")
    print("-> Saved model weight: ppo_double_lstm_scaler.pth")
    
    # 2. Vẽ và xuất biểu đồ xung lực Reward
    plt.figure(figsize=(10, 5))
    plt.plot(reward_history, color='#1f77b4', linewidth=1.5, label='Cumulative Reward')
    # Thêm đường trung bình trượt 50 chặng để thấy rõ xu hướng hội tụ
    if len(reward_history) >= 50:
        smoothed_reward = np.convolve(reward_history, np.ones(50)/50, mode='valid')
        plt.plot(range(49, len(reward_history)), smoothed_reward, color='#ff7f0e', linewidth=2, label='Moving Avg (50)')
    plt.title('Convergence Curve: Cumulative Reward Over Episodes', fontsize=12, fontweight='bold')
    plt.xlabel('Episode', fontsize=10)
    plt.ylabel('Total Reward', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.savefig('ppo_reward_convergence.pdf', bbox_inches='tight')
    plt.savefig('ppo_reward_convergence.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("-> Saved reward plot: ppo_reward_convergence.pdf/.png")

    # 3. Vẽ và xuất biểu đồ hội tụ hàm Loss
    plt.figure(figsize=(10, 5))
    plt.plot(loss_history, color='#d62728', linewidth=1.2, alpha=0.4, label='Total Loss (Raw)')
    # Thêm đường trung bình trượt 10 chặng cho Loss
    if len(loss_history) >= 10:
        smoothed_loss = np.convolve(loss_history, np.ones(10)/10, mode='valid')
        plt.plot(range(9, len(loss_history)), smoothed_loss, color='#d62728', linewidth=2, label='Total Loss (Smoothed)')
    plt.title('Model Optimization: Total PPO Loss Over Episodes', fontsize=12, fontweight='bold')
    plt.xlabel('Episode', fontsize=10)
    plt.ylabel('Loss Value', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.savefig('ppo_loss_convergence.pdf', bbox_inches='tight')
    plt.savefig('ppo_loss_convergence.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("-> Saved loss plot: ppo_loss_convergence.pdf/.png")

    # 4. Vẽ biểu đồ các thành phần Loss chi tiết (Mới bổ sung để báo cáo trực quan)
    plt.figure(figsize=(12, 6))
    plt.subplot(3, 1, 1)
    plt.plot(actor_loss_history, color='purple', label='Actor Loss')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.title('Detailed Loss Components Analysis')
    
    plt.subplot(3, 1, 2)
    plt.plot(critic_loss_history, color='orange', label='Critic Loss')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    
    plt.subplot(3, 1, 3)
    plt.plot(entropy_history, color='green', label='Entropy')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.xlabel('Episode')
    
    plt.tight_layout()
    plt.savefig('ppo_detailed_losses.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("-> Saved detailed loss components plot: ppo_detailed_losses.png")

if __name__ == "__main__":
    train()


In [ ]:
# Kiểm tra danh sách file hiện có
!ls -l *.py

# Chạy huấn luyện nếu file đã tồn tại
import os
if os.path.exists('train.py'):
    !python train.py
else:
    print('Lỗi: Bạn chưa chạy các cell %%writefile ở trên để tạo file!')